# Задача 03. Топ-3 товара внутри каждой категории

**Постановка задачи:** бизнесу нужен список сильных товаров внутри каждой категории. Нужно найти топ-3 товара по выручке в каждой категории.

В решении использовать оконную функцию `RANK()` или `ROW_NUMBER()`. В расчёт брать только доставленные заказы.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
WITH product_revenue AS (
    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(oi.quantity) AS units_sold,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.status = 'delivered'
    GROUP BY p.category, p.product_id, p.product_name
), ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS revenue_rank
    FROM product_revenue
)
SELECT
    category,
    product_id,
    product_name,
    units_sold,
    ROUND(revenue, 2) AS revenue,
    revenue_rank
FROM ranked
WHERE revenue_rank <= 3
ORDER BY category, revenue_rank;
"""

df = pd.read_sql_query(query, conn)
df

**Комментарий стажёра:** оконная функция удобна тем, что ранжирует товары отдельно внутри каждой категории, а не по всей таблице сразу.